# SQL 3: NULL handling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-03-null-handling.ipynb)

`NULL` means the value is **unknown**. It is not zero, not an empty string, and not a valid answer. Because the
value is unknown, most comparisons with it are unknown too, and a filter keeps only the rows where the condition is
definitely true. Most NULL bugs in analytics come from forgetting that rule.

This chapter shows how NULL behaves in comparisons, filters, aggregates, sorting and `NOT IN`, and how to handle
it deliberately. It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'a@x.com', 10), (2, NULL, NULL), (3, 'b@x.com', 30)
AS contacts(id, email, score)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tiny_orders AS
SELECT * FROM VALUES (1), (NULL) AS tiny_orders(customer_id)
""")

### The data

- `contacts` has 3 rows. Contact 2 has no email and no score, so both columns are NULL in that row.
- `tiny_orders` has 2 rows: one order for customer 1 and one order whose `customer_id` is NULL.
- `customers` has 8 rows, customer ids 1 to 8.

Small tables make it possible to work every result out by hand before running the query.

In [ ]:
spark.sql("SELECT * FROM contacts ORDER BY id").show()
spark.sql("SELECT * FROM tiny_orders").show()

## 1. Three possible answers

A comparison in SQL returns **true**, **false** or **unknown**. This is called *three-valued logic*. The third value
is the one that surprises people.

Why unknown? `NULL` stands for a value nobody knows. Is `1 = NULL`? The hidden value might be 1 or might not, so the
honest answer is "unknown". Spark represents unknown as a NULL boolean, which `collect()` returns as `None`.

`WHERE` keeps only the rows where the condition is **true**. Rows where it is false *or unknown* are dropped.

### Where you'll use it

- **Three-valued logic in filters:** a `WHERE` clause that silently drops rows with a NULL column, a common cause of missing data in reports.
- **Data quality checks:** a test such as `WHERE amount < 0` cannot find rows where `amount` is NULL. Those need their own `IS NULL` check.

Work it out first:

| Expression | Answer | Why |
|---|---|---|
| `1 = 1` | true | both values are known and equal |
| `1 = NULL` | unknown | the right side is not known |
| `NULL = NULL` | unknown | two unknown values might differ |
| `NULL IS NULL` | true | `IS NULL` asks "is this value missing?", which has a definite answer |

In [ ]:
q = """
SELECT
  (1 = 1)         AS one_equals_one,
  (1 = NULL)      AS one_equals_null,
  (NULL = NULL)   AS null_equals_null,
  (NULL IS NULL)  AS is_null
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(True, None, None, True)]

### AND, OR and NOT with unknown

Unknown combines with true and false in a way you can reason out. Ask: "could the unknown value change the answer?"

- `unknown AND false` is **false**. One false side makes AND false, whatever the other side is.
- `unknown AND true` is **unknown**. The answer depends on the unknown side.
- `unknown OR true` is **true**. One true side makes OR true.
- `unknown OR false` is **unknown**.
- `NOT unknown` is **unknown**. Negating something you do not know tells you nothing new.

The last rule matters most: wrapping a condition in `NOT` does not bring back the rows it dropped.
`CAST(NULL AS BOOLEAN)` below is an unknown truth value.

In [ ]:
q = """
SELECT
  CAST(NULL AS BOOLEAN) AND false AS unknown_and_false,
  CAST(NULL AS BOOLEAN) AND true  AS unknown_and_true,
  CAST(NULL AS BOOLEAN) OR  true  AS unknown_or_true,
  CAST(NULL AS BOOLEAN) OR  false AS unknown_or_false,
  NOT CAST(NULL AS BOOLEAN)       AS not_unknown
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(False, None, True, None, None)]

### NULL-safe comparison

`NULL = NULL` is not true. To ask whether a value is missing, use `IS NULL`. To compare two values so that NULL
counts as equal to NULL, use a **NULL-safe** comparison. A NULL-safe comparison never returns unknown: it returns true
when both sides are NULL, false when only one side is NULL, and the ordinary answer otherwise.

Spark writes it as `<=>`. The standard SQL spelling is `IS NOT DISTINCT FROM`, and Spark accepts that too.
`IS DISTINCT FROM` is its negation.

In [ ]:
q = """
SELECT
  NULL <=> NULL                 AS both_null,
  1 <=> NULL                    AS one_null,
  1 <=> 1                       AS both_equal,
  1 IS DISTINCT FROM NULL       AS distinct_from_null,
  NULL IS NOT DISTINCT FROM NULL AS standard_spelling
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(True, False, True, True, True)]

## 2. Filtering with NULL

A row whose `email` is NULL does not satisfy `email <> 'a@x.com'`, because the answer is unknown. It disappears from
the result. If you want those rows, say so explicitly.

Work it out row by row:

| id | email | `email <> 'a@x.com'` | kept by `WHERE`? |
|---|---|---|---|
| 1 | a@x.com | false | no |
| 2 | NULL | unknown | no |
| 3 | b@x.com | true | yes |

Only contact 3 survives. Adding `email IS NULL OR ...` turns row 2's condition into `true OR unknown`, which is true.

### Where you'll use it

- **Finding missing data:** `WHERE shipped_at IS NULL` lists orders not yet shipped.
- **Exclusion filters:** `WHERE country <> 'US'` also drops every row whose country is unknown. Decide whether those rows belong in the result.

In [ ]:
naive = "SELECT id FROM contacts WHERE email <> 'a@x.com' ORDER BY id"
explicit = "SELECT id FROM contacts WHERE email IS NULL OR email <> 'a@x.com' ORDER BY id"
spark.sql(naive).show()
spark.sql(explicit).show()

assert spark.sql(naive).collect() == [(3,)]
assert spark.sql(explicit).collect() == [(2,), (3,)]

### NOT does not recover unknown rows

`email = 'a@x.com'` keeps contact 1. You might expect `NOT (email = 'a@x.com')` to keep the other two. It keeps only
contact 3, because for contact 2 it evaluates `NOT unknown`, which is still unknown. `IS DISTINCT FROM` treats NULL as
an ordinary value, so it keeps both.

In [ ]:
negated = "SELECT id FROM contacts WHERE NOT (email = 'a@x.com') ORDER BY id"
distinct = "SELECT id FROM contacts WHERE email IS DISTINCT FROM 'a@x.com' ORDER BY id"

assert spark.sql(negated).collect() == [(3,)]
assert spark.sql(distinct).collect() == [(2,), (3,)]

### NULL join keys never match

A join condition is a filter on pairs of rows, so the same rule applies. The order in `tiny_orders` with a NULL
`customer_id` compares as unknown against every customer, and an inner join drops it. A NULL-safe condition would
match NULL to NULL, but no customer has a NULL id, so the order still has no partner here.

In [ ]:
q = """
SELECT t.customer_id, c.name
FROM tiny_orders t
JOIN customers c ON c.customer_id = t.customer_id
"""
spark.sql(q).show()
assert spark.sql("SELECT COUNT(*) FROM tiny_orders").collect() == [(2,)]
assert spark.sql(q).collect() == [(1, "Acme Corp")]

## 3. Aggregates skip NULL

`COUNT(*)` counts rows. `COUNT(column)` counts the non-NULL values in a column. `SUM`, `AVG`, `MIN` and `MAX` ignore
NULL inputs, so the average score is taken over two rows, not three.

By hand: the scores are 10, NULL and 30. `COUNT(*)` is 3. `COUNT(score)` is 2. `AVG(score)` is
(10 + 30) / 2 = 20.0, not (10 + 30) / 3.

### Where you'll use it

- **Averages over partial data:** `AVG(discount)` ignores orders without a discount, which is correct only if that is what the metric means.
- **Completeness checks:** `COUNT(*) - COUNT(email)` is the number of rows missing an email, a common data quality metric.

In [ ]:
q = """
SELECT
  COUNT(*)     AS all_rows,
  COUNT(email) AS with_email,
  COUNT(score) AS with_score,
  AVG(score)   AS average_score
FROM contacts
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3, 2, 2, 20.0)]

### Skipping NULL changes the meaning of an average

If a missing score should count as 0, replace it before averaging. The result changes from 20.0 to
(10 + 0 + 30) / 3 = 13.33. Neither number is wrong; they answer different questions, so choose on purpose.

In [ ]:
q = "SELECT AVG(score) AS skip_null, AVG(COALESCE(score, 0)) AS null_as_zero FROM contacts"
spark.sql(q).show()
skip_null, null_as_zero = spark.sql(q).collect()[0]
assert skip_null == 20.0
assert abs(null_as_zero - 40 / 3) < 1e-9

### Aggregates over no rows

When no rows reach an aggregate, `COUNT` returns 0 but `SUM`, `AVG`, `MIN` and `MAX` return NULL: there is nothing to
add up, so the total is unknown rather than zero. The filter `id > 10` below matches no contact.

In [ ]:
q = "SELECT COUNT(*), COUNT(score), SUM(score), MAX(score) FROM contacts WHERE id > 10"
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, 0, None, None)]
assert spark.sql("SELECT COALESCE(SUM(score), 0) FROM contacts WHERE id > 10").collect() == [(0,)]

### NULL as a group key

`GROUP BY` and `DISTINCT` do not use `=` to compare keys. They put all NULLs into **one** group, as if NULL were
equal to NULL. So a column with values `a`, NULL, NULL, `a`, `b` forms three groups: NULL (2 rows), `a` (2 rows) and
`b` (1 row).

In [ ]:
q = """
SELECT k, COUNT(*) AS n
FROM VALUES ('a'), (NULL), (NULL), ('a'), ('b') AS t(k)
GROUP BY k
ORDER BY k
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(None, 2), ("a", 2), ("b", 1)]

## 4. COALESCE and NULLIF

`COALESCE(a, b, ...)` returns the first argument that is not NULL. It is how you supply a default.
`NULLIF(a, b)` returns NULL when `a` equals `b`, and `a` otherwise. It turns a placeholder such as `0` into a real NULL.

### Where you'll use it

- **Default values in reports:** showing "unknown" instead of a blank region.
- **Avoiding divide-by-zero:** `x / NULLIF(y, 0)` returns NULL instead of an error.
- **Cleaning placeholders:** `NULLIF(phone, '')` turns empty strings from a source system into NULL, so `COUNT(phone)` counts only real values.

### NULL in expressions

Arithmetic with NULL gives NULL, so one missing value makes the whole sum unknown. String functions behave
differently: `concat` returns NULL if any part is NULL, while `concat_ws` (concatenate with separator) skips the NULL
parts. This is why defaults matter: `price + COALESCE(tax, 0)` stays known when `tax` is missing.

In [ ]:
q = """
SELECT 1 + NULL                            AS arithmetic,
       1 + COALESCE(NULL, 0)               AS arithmetic_with_default,
       concat('a', NULL)                   AS concat_with_null,
       concat_ws('-', 'a', NULL, 'c')      AS concat_ws_skips_null
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(None, 1, None, "a-c")]

### COALESCE supplies a default

Contact 2 has no score, so `COALESCE(score, 0)` falls through to the second argument. The other rows keep their own
score. `COALESCE` can take more than two arguments: `COALESCE(mobile, office, 'none')` tries each in turn.

In [ ]:
q = """
SELECT id, COALESCE(score, 0) AS score_or_zero, COALESCE(email, 'none') AS email_or_none
FROM contacts
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 10, "a@x.com"), (2, 0, "none"), (3, 30, "b@x.com")]

### NULLIF removes a placeholder and guards division

`NULLIF(0, 0)` is NULL because the two arguments are equal. `NULLIF(5, 0)` is 5.

That makes `NULLIF` a guard for division. With ANSI mode on, as here, `10 / 0` raises a `DIVIDE_BY_ZERO` error and
stops the query. `10 / NULLIF(0, 0)` divides by NULL instead, and anything divided by NULL is NULL. Spark's
`try_divide(10, 0)` gives the same NULL.

In [ ]:
from pyspark.errors import ArithmeticException

q = "SELECT NULLIF(0, 0) AS placeholder_removed, NULLIF(5, 0) AS kept"
spark.sql(q).show()
assert spark.sql(q).collect() == [(None, 5)]

try:
    spark.sql("SELECT 10 / 0").collect()
    raise AssertionError("expected a divide-by-zero error")
except ArithmeticException:
    print("10 / 0 raised DIVIDE_BY_ZERO")

assert spark.sql("SELECT 10 / NULLIF(0, 0), try_divide(10, 0)").collect() == [(None, None)]

## 5. Sorting NULL

NULL has no position among real values, so each engine picks a convention. Spark treats NULL as smaller than any
value: it sorts **first** in ascending order and **last** in descending order. PostgreSQL and Oracle do the opposite.
Write `NULLS FIRST` or `NULLS LAST` when the position matters, which also makes the query behave the same on every
engine.

Scores are 10 (contact 1), NULL (contact 2) and 30 (contact 3), so ascending order is 2, 1, 3.

### Where you'll use it

- **Dashboards:** deciding whether missing values sort first or last, so the top of a list is the data people want.
- **Top-N queries:** `ORDER BY revenue DESC NULLS LAST LIMIT 10` keeps rows with unknown revenue out of the top ten on every engine.

In [ ]:
asc_default = "SELECT id FROM contacts ORDER BY score, id"
asc_nulls_last = "SELECT id FROM contacts ORDER BY score NULLS LAST, id"
desc_default = "SELECT id FROM contacts ORDER BY score DESC, id"
spark.sql(asc_default).show()
spark.sql(asc_nulls_last).show()

assert spark.sql(asc_default).collect() == [(2,), (1,), (3,)]
assert spark.sql(asc_nulls_last).collect() == [(1,), (3,), (2,)]
assert spark.sql(desc_default).collect() == [(3,), (1,), (2,)]

## 6. The NOT IN trap

`x NOT IN (list)` is true only when `x` differs from **every** item in the list. If the list contains a NULL, that
comparison is unknown, so no row can be proven to pass and the query returns nothing. `NOT EXISTS` compares one row
at a time, so the NULL simply matches nobody.

Work it out for customer 2 against the list `(1, NULL)` from `tiny_orders`:

```text
2 NOT IN (1, NULL)
= NOT (2 = 1 OR 2 = NULL)
= NOT (false OR unknown)
= NOT unknown
= unknown            -> row dropped
```

Customer 1 is dropped too, correctly, because `1 = 1` is true. Every customer is dropped, so the result is empty.

### Where you'll use it

- **Anti-joins in pipelines:** finding customers with no orders. `NOT IN` returns no rows as soon as the subquery contains a NULL, so warehouses use `NOT EXISTS` instead.

In [ ]:
not_in = "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)"
not_exists = """
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
ORDER BY c.customer_id
"""
spark.sql(not_in).show()
spark.sql(not_exists).show()

assert spark.sql(not_in).collect() == []
assert spark.sql(not_exists).collect() == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]

### IN is safe, and NOT IN can be repaired

`IN` does not have the problem: `1 IN (1, NULL)` is `true OR unknown`, which is true. Only the negated form breaks.
If you must use `NOT IN`, remove the NULLs from the subquery first. Then the list is `(1)` and the logic is ordinary.

In [ ]:
in_query = "SELECT customer_id FROM customers WHERE customer_id IN (SELECT customer_id FROM tiny_orders)"
repaired = """
SELECT customer_id FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders WHERE customer_id IS NOT NULL)
ORDER BY customer_id
"""
assert spark.sql(in_query).collect() == [(1,)]
assert spark.sql(repaired).collect() == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]

## Interview notes

- **Three-valued logic.** Every comparison with NULL is unknown, and `WHERE` keeps only true rows. Most NULL bugs are
  a filter that silently drops unknown rows. `NOT` of unknown is still unknown.
- **NULL keys in joins** never match, so rows with a NULL key disappear from an inner join. Check the join key's
  NULL count before you join.
- **Grouping is different.** `GROUP BY` and `DISTINCT` put all NULLs in one group, even though `NULL = NULL` is unknown.
- **Sum of nothing is NULL, count of nothing is 0.** Wrap aggregates in `COALESCE` when a report needs zero.
- **`NOT IN` with a NULL returns nothing.** Use `NOT EXISTS`, or filter NULLs out of the subquery.
- **NULL sort order differs by engine.** Spark sorts NULL first ascending; PostgreSQL sorts it last. Write
  `NULLS FIRST` or `NULLS LAST` explicitly.

## Summary

- `= NULL` is never true. Use `IS NULL`, or `<=>` (`IS NOT DISTINCT FROM`) for a NULL-safe comparison.
- `COUNT(column)` and `COUNT(*)` answer different questions. `SUM`, `AVG`, `MIN` and `MAX` skip NULL.
- Use `COALESCE` for defaults, and `NULLIF` to remove placeholders and guard division.
- State `NULLS FIRST` or `NULLS LAST` when the position of missing values matters.
- Prefer `NOT EXISTS` to `NOT IN` when the other side can contain NULL.

## Practice questions

1. How many contacts have no email?
2. What is the average score, ignoring contacts without a score?
3. Which contact ids have an email that is not `'a@x.com'`, including contacts whose email is unknown? Sort by id.
4. Which customers do not appear in `tiny_orders`? Use `NOT EXISTS`, sorted by `customer_id`.
5. What is the average score if a missing score counts as 0?
6. List contact ids from the highest score to the lowest, with contacts that have no score at the end.
7. Rewrite question 4 with `NOT IN` so that it returns the right answer.

## Answers

### Question 1

`IS NULL` is the only test that is true for a missing value. `COUNT(*) - COUNT(email)` gives the same answer.

In [ ]:
q = "SELECT COUNT(*) AS missing_email FROM contacts WHERE email IS NULL"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1,)]
assert spark.sql("SELECT COUNT(*) - COUNT(email) FROM contacts").collect() == [(1,)]

### Question 2

`AVG` already skips NULL, so no filter is needed: (10 + 30) / 2 = 20.0.

In [ ]:
q = "SELECT AVG(score) AS average_score FROM contacts"
spark.sql(q).show()
assert spark.sql(q).collect() == [(20.0,)]

### Question 3

`email <> 'a@x.com'` is unknown for contact 2, so add `email IS NULL` with `OR` to keep it.

In [ ]:
q = "SELECT id FROM contacts WHERE email IS NULL OR email <> 'a@x.com' ORDER BY id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (3,)]

### Question 4

`NOT EXISTS` checks each customer separately. The NULL order matches no customer, so it does not affect the result.

In [ ]:
q = """
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]

### Question 5

Replace NULL with 0 before averaging, so all three rows count: (10 + 0 + 30) / 3 = 13.33.

In [ ]:
q = "SELECT AVG(COALESCE(score, 0)) AS average_score FROM contacts"
spark.sql(q).show()
assert abs(spark.sql(q).collect()[0][0] - 40 / 3) < 1e-9

### Question 6

Spark already puts NULL last in descending order, but writing `NULLS LAST` makes the intent explicit and portable.

In [ ]:
q = "SELECT id FROM contacts ORDER BY score DESC NULLS LAST, id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(3,), (1,), (2,)]

### Question 7

Filter the NULL out of the subquery. The list becomes `(1)`, and `NOT IN` then behaves as expected.

In [ ]:
q = """
SELECT customer_id
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders WHERE customer_id IS NOT NULL)
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]